# Team-6 전체 파이프라인 통합 노트북

이 노트북은 프로젝트의 단계별 노트북을 하나로 합친 파일입니다. 원본 노트북은 `notebooks/01_...`부터 `notebooks/04_...`까지 단계별 폴더에 그대로 유지됩니다.


# 1단계 EDA 및 도시별 샘플링

Yelp 원본 JSON에서 분석 대상 도시의 레스토랑 리뷰를 추출하고, 감성 라벨을 만든 뒤 도시별 15,000건 샘플 CSV를 저장합니다. 마지막 섹션에서는 샘플 품질을 확인하는 표와 그래프를 `output/`에 함께 저장합니다.

노트북 위치: `notebooks/01_eda_sampling/`


In [ ]:
from google.colab import drive
from google.colab import files
from pathlib import Path

# 1. 내 구글 드라이브 연동 (결과물을 저장하기 위함)
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
FIGURE_DIR = PROJECT_ROOT / 'output' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
for directory in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# 2. 캐글 API 토큰(kaggle.json) 업로드
print("캐글에서 다운받은 kaggle.json 파일을 업로드해주세요.")
files.upload()

## 1. Colab 및 Kaggle 환경 준비

구글 드라이브를 마운트하고 Kaggle API 토큰을 등록합니다. 원본 JSON은 코랩 임시 저장소에 압축 해제하여 읽기 속도를 확보합니다.


In [ ]:
# 캐글 폴더 세팅 및 권한 부여
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Yelp 데이터셋 다운로드 (약 1~2분 소요)
!kaggle datasets download -d yelp-dataset/yelp-dataset

# 코랩 임시 공간에 압축 풀기 (드라이브에 푸는 것보다 수십 배 빠름)
!unzip -q yelp-dataset.zip -d /content/yelp_data
print("데이터 다운로드 및 압축 해제 완료!")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 필라델피아 도시 필터링
target_city = 'Philadelphia'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist()) # 검색 속도를 위해 set으로 변환
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

## 2. Philadelphia 샘플 생성

비즈니스 JSON에서 Philadelphia 레스토랑 `business_id`를 먼저 추출한 뒤, 리뷰 JSON을 청크 단위로 읽어 해당 식당 리뷰만 남깁니다.


In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

print("리뷰 데이터 청크 필터링 시작... (약 3~5분 소요 예상)")

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링 (10,000개 이상 요건 충족)
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)
display(final_dataset.head(3))

In [ ]:
# 내 구글 드라이브의 원하는 경로 지정 (예: Colab Notebooks 폴더)
save_path = INTERIM_DIR / 'yelp_subset_philly_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")
print("코랩 세션이 초기화되어도 저장된 CSV에서 전처리를 이어갈 수 있습니다.")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 투손(Tucson) 도시 필터링 적용
target_city = 'Tucson'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist())
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

print(f"{target_city} 리뷰 데이터 청크 필터링 시작... (약 3~5분 소요 예상)")

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

## 3. Tucson 샘플 생성

동일한 기준으로 Tucson 레스토랑 리뷰를 필터링하고 3점 중립 리뷰를 제거한 뒤 15,000건을 샘플링합니다.


In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)

display(final_dataset.head(3))

In [ ]:
# 내 구글 드라이브의 원하는 경로 지정 (Tucson 파일명 적용)
save_path = INTERIM_DIR / 'yelp_subset_tucson_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")
print("투손 지역 15,000개 리뷰 데이터를 저장했습니다.")

In [ ]:
import pandas as pd

# 비즈니스 데이터 불러오기
biz_path = '/content/yelp_data/yelp_academic_dataset_business.json'
biz_df = pd.read_json(biz_path, lines=True)

# 'Restaurants' 카테고리 필터링 (결측치 처리 포함)
biz_df = biz_df[biz_df['categories'].fillna('').str.contains('Restaurants')]

# 뉴올리언스(New Orleans) 도시 필터링 적용
target_city = 'New Orleans'
biz_city = biz_df[biz_df['city'] == target_city]

# 추출할 식당들의 고유 ID 리스트 확보
target_biz_ids = set(biz_city['business_id'].tolist())
print(f"{target_city} 내 대상 식당 개수: {len(target_biz_ids)}개")

In [ ]:
review_path = '/content/yelp_data/yelp_academic_dataset_review.json'
chunk_size = 100000
review_chunks = []

print(f"{target_city} 리뷰 데이터 청크 필터링 시작...")

# 10만 줄씩 잘라서 읽기
for chunk in pd.read_json(review_path, lines=True, chunksize=chunk_size):
    # 대상 식당 ID에 해당하는 리뷰만 남기기
    filtered_chunk = chunk[chunk['business_id'].isin(target_biz_ids)]
    review_chunks.append(filtered_chunk)

# 모아둔 조각들을 하나의 데이터프레임으로 병합
review_subset = pd.concat(review_chunks, ignore_index=True)
print(f"필터링된 총 리뷰 개수: {len(review_subset)}개")

In [ ]:
# 1. 3점 리뷰(중립) 제거
review_subset = review_subset[review_subset['stars'] != 3]

# 2. 4~5점은 1(긍정), 1~2점은 0(부정)으로 매핑
review_subset['is_positive'] = review_subset['stars'].apply(lambda x: 1 if x > 3 else 0)

# 3. 15,000개 무작위 샘플링
final_dataset = review_subset.sample(n=15000, random_state=42)

print("최종 샘플링 데이터 크기:", final_dataset.shape)
display(final_dataset.head(3))

## 4. New Orleans 샘플 생성

New Orleans도 같은 파이프라인을 적용하여 도시별 비교가 가능한 입력 CSV를 만듭니다.


In [ ]:
# 내 구글 드라이브의 원하는 경로 지정 (New Orleans 파일명 적용)
save_path = INTERIM_DIR / 'yelp_subset_new_orleans_15k.csv'

# CSV 파일로 영구 저장
final_dataset.to_csv(save_path, index=False)

print(f"저장 완료. 경로: {save_path}")
print("뉴올리언스 지역 15,000개 리뷰 데이터를 저장했습니다.")

In [ ]:
# 1. 코랩 리눅스 시스템에 나눔 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 2. 설치된 폰트를 Matplotlib의 기본 폰트로 설정
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

# 3. 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

## 5. 샘플 EDA 및 시각화 저장

각 도시 샘플의 타깃 분포, 텍스트 길이, 워드클라우드를 확인하고 PNG 파일을 `output/figures/`에 저장합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

BASE_STOPWORDS = {
    "food", "place", "restaurant", "good", "great", "time", "one",
    "really", "even", "came", "went", "got", "make", "will", "go",
    "ordered", "us", "back", "much", "well"
}


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print(f"그래프 저장 완료: {path}")


def plot_city_eda(city_name, city_slug, input_file, extra_stopwords=None):
    file_path = INTERIM_DIR / input_file
    df = pd.read_csv(file_path)
    df['text_length'] = df['text'].astype(str).apply(len)

    summary_table = df.groupby('is_positive').agg(
        review_count=('review_id', 'count'),
        avg_stars=('stars', 'mean'),
        avg_text_length=('text_length', 'mean'),
    ).reset_index()
    summary_path = TABLE_DIR / f'eda_{city_slug}_summary.csv'
    summary_table.to_csv(summary_path, index=False)
    print(f"[{city_name}] 요약표 저장 완료: {summary_path}")
    display(summary_table)

    plt.figure(figsize=(8, 6))
    ax = sns.countplot(data=df, x='is_positive', hue='is_positive', palette=['#ff9999', '#66b3ff'], legend=False)
    plt.title(f'{city_name} 리뷰 만족도 분포', fontsize=16, fontweight='bold')
    plt.xlabel('만족도 (0: 불만족, 1: 만족)', fontsize=12)
    plt.ylabel('리뷰 개수', fontsize=12)
    for patch in ax.patches:
        height = patch.get_height()
        ax.text(patch.get_x() + patch.get_width() / 2, height + 50, f'{int(height):,}', ha='center', size=12)
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_target_distribution.png')
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='is_positive', y='text_length', hue='is_positive', palette=['#ff9999', '#66b3ff'], legend=False)
    plt.title(f'{city_name} 만족도별 리뷰 텍스트 길이', fontsize=16, fontweight='bold')
    plt.xlabel('만족도 (0: 불만족, 1: 만족)', fontsize=12)
    plt.ylabel('리뷰 길이 (글자 수)', fontsize=12)
    plt.ylim(0, df['text_length'].quantile(0.95))
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_text_length_boxplot.png')
    plt.show()

    pos_text = " ".join(review for review in df[df['is_positive'] == 1]['text'].astype(str))
    neg_text = " ".join(review for review in df[df['is_positive'] == 0]['text'].astype(str))
    custom_stopwords = set(STOPWORDS) | BASE_STOPWORDS | set(extra_stopwords or [])

    wc_pos = WordCloud(width=600, height=400, background_color='white', colormap='Greens', stopwords=custom_stopwords, max_words=100).generate(pos_text)
    wc_neg = WordCloud(width=600, height=400, background_color='white', colormap='Reds', stopwords=custom_stopwords, max_words=100).generate(neg_text)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('만족(긍정) 핵심 키워드', fontsize=18, fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('불만족(부정) 위기 키워드', fontsize=18, fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    save_figure(f'eda_{city_slug}_wordcloud.png')
    plt.show()


### 필라델피아 EDA

샘플의 타깃 불균형, 텍스트 길이, 긍정/부정 키워드를 확인합니다.


In [ ]:
plot_city_eda('Philadelphia', 'philly', 'yelp_subset_philly_15k.csv')

### 투손 EDA

도시명이 워드클라우드에 과도하게 반영되지 않도록 Tucson 관련 단어를 stopword에 추가합니다.


In [ ]:
plot_city_eda('Tucson', 'tucson', 'yelp_subset_tucson_15k.csv', extra_stopwords=['Tucson', 'tucson'])

### 뉴올리언스 EDA

New Orleans/NOLA 표현이 핵심 키워드를 가리지 않도록 도시명 관련 단어를 제외합니다.


In [ ]:
plot_city_eda('New Orleans', 'new_orleans', 'yelp_subset_new_orleans_15k.csv', extra_stopwords=['New Orlean', 'New Orleans', 'NOLA', 'nola', 'new orleans', 'New', 'Orlean'])

---

# 2-1단계. 사업장 위치 CSV 생성

원본 사업장 JSON에서 위치 정보만 추출해 `data/interim/business_location.csv`를 만듭니다.

원본 노트북: `notebooks/02_feature_engineering/Create_Business_Location_CSV.ipynb`


# 사업장 위치 CSV 생성

원본 `yelp_academic_dataset_business.json`에서 피처 엔지니어링에 필요한 위치 정보만 추출합니다.

생성 파일:

```text
business_location.csv
```

포함 컬럼:

```text
business_id, city, state, latitude, longitude
```

## 1. 경로 설정

로컬에서는 기본적으로 `team_project/archive/yelp_academic_dataset_business.json`을 찾습니다. Colab에서 실행하려면 Google Drive에 JSON을 올린 뒤 `BUSINESS_JSON_PATH`만 해당 경로로 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import pandas as pd


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False


IN_COLAB = running_in_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
    BUSINESS_JSON_PATH = PROJECT_ROOT / 'data' / 'yelp_academic_dataset_business.json'
else:
    PROJECT_ROOT = Path.cwd().resolve()
    for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if path.name == 'Team-6':
            PROJECT_ROOT = path
            break

    BUSINESS_JSON_PATH = PROJECT_ROOT.parent / 'archive' / 'yelp_academic_dataset_business.json'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'interim'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'business_location.csv'

print(f'코랩 실행 여부: {IN_COLAB}')
print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'사업장 JSON 경로: {BUSINESS_JSON_PATH}')
print(f'저장 경로: {OUTPUT_PATH}')

## 2. 원본 JSON 확인

In [ ]:
if not BUSINESS_JSON_PATH.exists():
    raise FileNotFoundError(
        '사업장 JSON 파일을 찾을 수 없습니다. '\
        'BUSINESS_JSON_PATH를 실제 파일 경로로 수정해 주세요.\n'
        f'BUSINESS_JSON_PATH: {BUSINESS_JSON_PATH}'
    )

print('사업장 JSON 파일 확인 완료')
print(f'파일 크기(MB): {BUSINESS_JSON_PATH.stat().st_size / 1024 / 1024:.2f}')

## 3. 위치 정보 추출

In [ ]:
USE_COLUMNS = ['business_id', 'city', 'state', 'latitude', 'longitude']

business_df = pd.read_json(BUSINESS_JSON_PATH, lines=True)
business_location = business_df[USE_COLUMNS].copy()

business_location = business_location.dropna(subset=['business_id'])
business_location = business_location.drop_duplicates('business_id')

print('사업장 위치 데이터 크기:', business_location.shape)
display(business_location.head())

## 4. 저장 및 검증

In [ ]:
business_location.to_csv(OUTPUT_PATH, index=False)

saved_df = pd.read_csv(OUTPUT_PATH)
print(f'저장 완료: {OUTPUT_PATH}')
print('저장된 데이터 크기:', saved_df.shape)
print('결측치 개수:')
print(saved_df.isnull().sum())
display(saved_df.head())

---

# 2-2단계. 피처 엔지니어링

도시별 샘플 CSV에 텍스트, 날짜, 유저, 식당, 위치 기반 파생변수를 추가합니다.

원본 노트북: `notebooks/02_feature_engineering/Feature_Engineering.ipynb`


# 2단계 피처 엔지니어링

1단계에서 생성한 도시별 샘플 CSV를 Google Drive에서 불러오고, 텍스트/날짜/유저/식당/상권 클러스터 파생변수를 추가한 뒤 `*_features.csv` 파일로 저장합니다.

이 노트북은 Colab 실행을 기준으로 작성되었습니다.

## 1. 환경 설정 및 경로 지정

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
if not PROJECT_ROOT.exists():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다. Team-6 저장소 안에서 노트북을 실행해 주세요.')

def find_input_dir(project_root):
    candidates = [
        project_root / 'data' / 'interim',
    ]
    for candidate in candidates:
        if (candidate / 'yelp_subset_philly_15k.csv').exists():
            print(f'입력 폴더 확인: {candidate}')
            return candidate

    matches = list(project_root.rglob('yelp_subset_philly_15k.csv'))
    if matches:
        detected_dir = matches[0].parent
        print(f'입력 폴더 확인: {detected_dir}')
        return detected_dir

    raise FileNotFoundError(f'프로젝트 경로 아래에서 yelp_subset_philly_15k.csv를 찾을 수 없습니다: {project_root}')


INPUT_DIR = find_input_dir(PROJECT_ROOT)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
REPORT_DIR = PROJECT_ROOT / 'output' / 'reports'
for directory in [OUTPUT_DIR, TABLE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project')
BUSINESS_LOCATION_CSV_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'interim' / 'business_location.csv',
    PROJECT_ROOT / 'data' / 'business_location.csv',
]

BUSINESS_JSON_CANDIDATES = [
    Path('/content/yelp_data/yelp_academic_dataset_business.json'),
    PROJECT_ROOT / 'data' / 'yelp_academic_dataset_business.json',
]

CITY_CONFIGS = [
    {
        'city': 'Philadelphia',
        'input_file': 'yelp_subset_philly_15k.csv',
        'output_file': 'yelp_subset_philly_15k_features.csv',
    },
    {
        'city': 'Tucson',
        'input_file': 'yelp_subset_tucson_15k.csv',
        'output_file': 'yelp_subset_tucson_15k_features.csv',
    },
    {
        'city': 'New Orleans',
        'input_file': 'yelp_subset_new_orleans_15k.csv',
        'output_file': 'yelp_subset_new_orleans_15k_features.csv',
    },
]

KMEANS_CLUSTERS = 5
RANDOM_STATE = 42

print(f'입력 폴더: {INPUT_DIR}')
print(f'출력 폴더: {OUTPUT_DIR}')
print(f'표 저장 폴더: {TABLE_DIR}')
print(f'보고서 저장 폴더: {REPORT_DIR}')

## 2. 입력 파일 확인

In [ ]:
def check_input_files(city_configs=CITY_CONFIGS):
    missing_files = []
    for config in city_configs:
        input_path = INPUT_DIR / config['input_file']
        if not input_path.exists():
            missing_files.append(input_path)

    if missing_files:
        missing_text = '\n'.join(str(path) for path in missing_files)
        raise FileNotFoundError(f'입력 CSV 파일을 찾을 수 없습니다:\n{missing_text}')

    print('입력 CSV 파일 확인 완료')


check_input_files()

## 3. 공통 함수 정의

In [ ]:
def add_text_features(df):
    df = df.copy()
    df['text'] = df['text'].fillna('').astype(str)

    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    df['sentence_count'] = df['text'].str.count(r'[.!?]+').clip(lower=1)
    df['avg_word_length'] = df['text_length'] / df['word_count'].clip(lower=1)
    df['uppercase_ratio'] = df['text'].apply(
        lambda text: sum(1 for char in text if char.isupper()) / max(len(text), 1)
    )
    df['exclamation_count'] = df['text'].str.count('!')
    df['question_count'] = df['text'].str.count(r'\?')
    df['engagement_sum'] = df[['useful', 'funny', 'cool']].fillna(0).sum(axis=1)
    return df


def add_date_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    df['review_year'] = df['date'].dt.year
    df['review_month'] = df['date'].dt.month
    df['review_dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['review_dayofweek'].isin([5, 6]).astype(int)
    return df


def add_user_features(df):
    df = df.copy()
    user_stats = df.groupby('user_id')['stars'].agg(
        user_review_count='count',
        user_avg_stars='mean'
    ).reset_index()

    df = df.merge(user_stats, on='user_id', how='left')
    df['user_star_deviation'] = df['stars'] - df['user_avg_stars']
    return df


def add_business_features(df):
    df = df.copy()
    business_stats = df.groupby('business_id')['stars'].agg(
        business_review_count='count',
        business_avg_stars='mean'
    ).reset_index()

    df = df.merge(business_stats, on='business_id', how='left')
    df['business_star_deviation'] = df['stars'] - df['business_avg_stars']
    return df


def find_business_location_csv_path():
    for path in BUSINESS_LOCATION_CSV_CANDIDATES:
        if path.exists():
            print(f'사업장 위치 CSV 확인: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('business_location.csv'))
    if matches:
        detected_path = matches[0]
        print(f'사업장 위치 CSV 확인: {detected_path}')
        return detected_path

    return None


def find_business_json_path():
    for path in BUSINESS_JSON_CANDIDATES:
        if path.exists():
            print(f'사업장 JSON 확인: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('yelp_academic_dataset_business.json'))
    if matches:
        detected_path = matches[0]
        print(f'사업장 JSON 확인: {detected_path}')
        return detected_path

    return None


def load_business_locations(business_ids):
    location_csv_path = find_business_location_csv_path()
    if location_csv_path is not None:
        location_df = pd.read_csv(location_csv_path)
        return location_df.loc[
            location_df['business_id'].isin(business_ids),
            ['business_id', 'latitude', 'longitude']
        ].drop_duplicates('business_id')

    business_json_path = find_business_json_path()
    if business_json_path is None:
        print('사업장 위치 CSV 또는 business.json을 찾지 못해 위치 기반 피처 생성을 건너뜁니다.')
        print('필요 파일명: business_location.csv 또는 yelp_academic_dataset_business.json')
        print(f'검색한 경로: {DRIVE_PROJECT_ROOT}')
        return None

    print(f'사업장 위치 정보 불러오기: {business_json_path}')
    biz_df = pd.read_json(business_json_path, lines=True)
    location_df = biz_df.loc[
        biz_df['business_id'].isin(business_ids),
        ['business_id', 'latitude', 'longitude']
    ].drop_duplicates('business_id')
    return location_df


def add_location_features(df, location_df):
    df = df.copy()
    if location_df is None:
        df['latitude'] = np.nan
        df['longitude'] = np.nan
        return df

    return df.merge(location_df, on='business_id', how='left')


def add_kmeans_cluster(df, n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE):
    df = df.copy()
    df['business_cluster'] = np.nan

    coords = df[['latitude', 'longitude']].dropna()
    if coords.empty:
        print('사용 가능한 좌표가 없어 K-Means 클러스터링을 건너뜁니다.')
        return df

    unique_count = coords.drop_duplicates().shape[0]
    cluster_count = min(n_clusters, unique_count)
    if cluster_count < 2:
        print('고유 좌표 수가 부족해 K-Means 클러스터링을 건너뜁니다.')
        return df

    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

    kmeans = KMeans(n_clusters=cluster_count, random_state=random_state, n_init='auto')
    df.loc[coords.index, 'business_cluster'] = kmeans.fit_predict(coords_scaled)
    df['business_cluster'] = df['business_cluster'].astype('Int64')
    return df


def add_all_features(df, location_df):
    df = add_text_features(df)
    df = add_date_features(df)
    df = add_user_features(df)
    df = add_business_features(df)
    df = add_location_features(df, location_df)
    df = add_kmeans_cluster(df)
    return df


def summarize_features(df):
    print('데이터 크기:', df.shape)
    print('\n타깃 분포:')
    print(df['is_positive'].value_counts(dropna=False))
    print('\n결측치가 많은 컬럼:')
    print(df.isnull().sum().sort_values(ascending=False).head(15))

## 4. 위치 정보 로드

`business.json`이 `/content/yelp_data` 또는 Drive의 `Team-6/data`에 있으면 위도/경도를 병합하고 K-Means 클러스터를 생성합니다. 파일이 없으면 위치 기반 피처만 건너뜁니다.

In [ ]:
all_business_ids = set()
for config in CITY_CONFIGS:
    temp_df = pd.read_csv(INPUT_DIR / config['input_file'], usecols=['business_id'])
    all_business_ids.update(temp_df['business_id'].dropna().unique())

location_df = load_business_locations(all_business_ids)
if location_df is not None:
    print('위치 정보 행 수:', location_df.shape[0])
    display(location_df.head())

## 5. 도시별 피처 생성 및 저장

In [ ]:
feature_datasets = {}

for config in CITY_CONFIGS:
    city = config['city']
    input_path = INPUT_DIR / config['input_file']
    output_path = OUTPUT_DIR / config['output_file']

    print(f'\n[{city}] 피처 생성 시작')
    df = pd.read_csv(input_path)
    print('원본 데이터 크기:', df.shape)

    feature_df = add_all_features(df, location_df)
    summarize_features(feature_df)

    feature_df.to_csv(output_path, index=False)
    feature_datasets[city] = feature_df
    print(f'저장 완료: {output_path}')

print('\n전체 피처 생성 완료')

## 6. 결과 확인

In [ ]:
for config in CITY_CONFIGS:
    output_path = OUTPUT_DIR / config['output_file']
    print(output_path, output_path.exists())

sample_city = CITY_CONFIGS[0]['city']
display(feature_datasets[sample_city].head())

## 7. 최종 CSV 저장 및 산출물 인덱스

최종 feature CSV를 `data/processed/`에 저장하고, 도시별 행 수/컬럼 수/파일 크기를 `output/tables/feature_engineering_outputs.csv`로 남깁니다.


In [ ]:
saved_outputs = []

for config in CITY_CONFIGS:
    city = config['city']
    output_path = OUTPUT_DIR / config['output_file']

    if city not in feature_datasets:
        raise KeyError(f"feature_datasets에서 {city} 데이터를 찾을 수 없습니다. 피처 생성 셀을 먼저 실행해 주세요.")

    feature_datasets[city].to_csv(output_path, index=False)
    saved_outputs.append({
        'city': city,
        'path': str(output_path),
        'rows': len(feature_datasets[city]),
        'columns': feature_datasets[city].shape[1],
        'file_size_mb': output_path.stat().st_size / 1024 / 1024,
    })

saved_outputs_df = pd.DataFrame(saved_outputs)
summary_path = TABLE_DIR / 'feature_engineering_outputs.csv'
saved_outputs_df.to_csv(summary_path, index=False)
display(saved_outputs_df)
print(f'최종 피처 CSV 저장 완료. 요약 파일: {summary_path}')


---

# 3단계. RoBERTa 임베딩 및 PCA

리뷰 텍스트를 RoBERTa 임베딩으로 변환하고 32차원 PCA CSV를 생성합니다.

원본 노트북: `notebooks/03_text_embedding/RoBERTa_PCA.ipynb`


## [3-1단계] 자연어 텍스트 임베딩 추출 (RoBERTa)

**1. 작업 목표**
머신러닝 모델(CatBoost)은 영어 문장을 그대로 읽을 수 없습니다. 따라서 사전 학습된 거대 언어 모델(LLM)을 활용하여, 리뷰 텍스트 문장의 전반적인 맥락과 감정을 컴퓨터가 이해할 수 있는 **768차원의 숫자 벡터** 로 번역하는 작업을 수행합니다.

**2. 세부 설정**
* **사용 모델:** `distilroberta-base` (RoBERTa 모델의 경량화 버전으로, 성능은 유지하면서 연산 속도가 훨씬 빠릅니다.)
* **하드웨어:** T4 GPU (연산 속도 최적화)
* **배치 처리:** GPU 메모리 초과(OOM) 에러를 방지하기 위해 15,000개의 데이터를 32개씩 쪼개어 순차적으로 임베딩을 추출합니다.

**3. 예상 결과물**
* `15,000(리뷰 개수) x 768(문맥 벡터)` 크기의 거대한 숫자 행렬 도출

## 저장 경로 규칙

노트북 파일은 `notebooks/03_text_embedding/`에 두고, 실행 결과인 `.npy` 임베딩과 PCA CSV는 `data/embeddings/`에 저장합니다. 이렇게 하면 코드와 데이터 산출물이 섞이지 않습니다.


## [3-1단계] RoBERTa 임베딩 추출: 필라델피아
* **목표:** 필라델피아 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **입력 데이터:** `yelp_subset_philly_15k.csv`
* **출력 데이터:** `philly_embeddings.npy` (구글 드라이브 자동 저장)

In [ ]:
!pip install transformers torch tqdm

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# 1. 필라델피아 데이터 로드
file_path = '/content/yelp_subset_philly_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 모델 로드
model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[필라델피아] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

# 4. 행렬 병합 및 드라이브 안전 저장
philly_embeddings = np.vstack(embeddings_list)
save_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/philly_embeddings.npy'
np.save(save_path, philly_embeddings)

print(f"필라델피아 임베딩 완료 및 드라이브 저장 완료. (크기: {philly_embeddings.shape})")

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

# 1. 혹시 풀렸을지 모를 구글 드라이브 다시 연결
drive.mount('/content/drive')

# 2. 코랩 내부 저장소(안전지대)에 먼저 파일 저장
local_path = '/content/philly_embeddings.npy'
np.save(local_path, philly_embeddings)
print("1. 코랩 내부 저장 완료.")

# 3. 팀 공유 구글 드라이브 폴더로 파일 복사
drive_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/philly_embeddings.npy'
Path(drive_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_path, drive_path)
print("2. 구글 드라이브 공유 문서함으로 복사 완료.")

## [3-2단계] PCA 차원 축소: 필라델피아
* **목표:** CatBoost 모델의 학습 속도와 효율을 높이기 위해, 필라델피아 데이터의 768차원 문맥 벡터를 핵심 정보만 남기고 32차원으로 압축합니다.
* **출력 데이터:** `philly_pca_32.csv` (`data/embeddings` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil
from pathlib import Path

print("[필라델피아] PCA 차원 축소 시작")

# 1. 32차원으로 압축하는 PCA 모델 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 구출해서 메모리에 살아있는 philly_embeddings 데이터를 바로 압축!
embeddings_pca = pca.fit_transform(philly_embeddings)

# 3. 데이터 손실률 및 크기 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보의 약 {explained_variance:.2f}%를 보존했습니다.")
print(f"압축된 데이터 크기: {embeddings_pca.shape} (15000개의 리뷰가 각각 32개의 숫자로 변환됨)")

# 4. 모델링 팀이 쓰기 좋게 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 코랩 내부에 먼저 CSV로 저장 후 구글 드라이브(text_embedding 폴더)로 복사
local_csv_path = '/content/philly_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/philly_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
Path(drive_csv_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_csv_path, drive_csv_path)

print(f"필라델피아 3단계 최종 CSV 저장 완료: {drive_csv_path}")

## [3-1단계] RoBERTa 임베딩 추출: 투손
* **목표:** 투손 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **출력 데이터:** `tucson_embeddings.npy` (`data/embeddings` 폴더에 저장)

In [ ]:
import pandas as pd
import numpy as np
import torch
import shutil
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from pathlib import Path
from google.colab import drive

# 1. 드라이브 마운트 및 연산 장치 확인
drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 연산 장치: {device}")

# 2. 데이터 및 모델 로드
file_path = '/content/yelp_subset_tucson_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

print("[투손] RoBERTa 텍스트 임베딩 추출 시작")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[투손] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

tucson_embeddings = np.vstack(embeddings_list)

# 4. 코랩 내부 저장 후 드라이브 복사 (연결 끊김 에러 완벽 방지)
local_npy_path = '/content/tucson_embeddings.npy'
drive_npy_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/tucson_embeddings.npy'

np.save(local_npy_path, tucson_embeddings)
Path(drive_npy_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_npy_path, drive_npy_path)

print(f"투손 임베딩 완료 및 저장 완료. (크기: {tucson_embeddings.shape})")

## [3-2단계] PCA 차원 축소: 투손
* **목표:** CatBoost 모델의 학습을 위해 투손 데이터의 768차원 벡터를 32차원으로 압축
* **출력 데이터:** `tucson_pca_32.csv` (`data/embeddings` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil
from pathlib import Path

print("[투손] PCA 차원 축소 시작")

# 1. 32차원으로 압축하는 PCA 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 생성된 tucson_embeddings 데이터를 바로 압축
embeddings_pca = pca.fit_transform(tucson_embeddings)

# 3. 데이터 손실률 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보 보존율: {explained_variance:.2f}%")
print(f"압축된 데이터 크기: {embeddings_pca.shape}")

# 4. 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 안전 저장 (코랩 ➔ 구글 드라이브)
local_csv_path = '/content/tucson_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/tucson_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
Path(drive_csv_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_csv_path, drive_csv_path)

print(f"투손 3단계 최종 CSV 저장 완료: {drive_csv_path}")

## [3-1단계] RoBERTa 임베딩 추출: 뉴올리언스
* **목표:** 뉴올리언스 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **출력 데이터:** `new_orleans_embeddings.npy` (`data/embeddings` 폴더에 저장)

In [ ]:
import pandas as pd
import numpy as np
import torch
import shutil
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from pathlib import Path
from google.colab import drive

# 1. 드라이브 마운트 및 연산 장치 확인
drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 연산 장치: {device}")

# 2. 데이터 및 모델 로드
file_path = '/content/yelp_subset_new_orleans_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

print("[뉴올리언스] RoBERTa 텍스트 임베딩 추출 시작")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[뉴올리언스] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

new_orleans_embeddings = np.vstack(embeddings_list)

# 4. 코랩 내부 저장 후 드라이브 복사 (안전 장치)
local_npy_path = '/content/new_orleans_embeddings.npy'
drive_npy_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/new_orleans_embeddings.npy'

np.save(local_npy_path, new_orleans_embeddings)
Path(drive_npy_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_npy_path, drive_npy_path)

print(f"뉴올리언스 임베딩 완료 및 저장 완료. (크기: {new_orleans_embeddings.shape})")

## [3-2단계] PCA 차원 축소: 뉴올리언스
* **목표:** CatBoost 모델의 학습 속도 향상을 위해 뉴올리언스 데이터의 768차원 벡터를 32차원으로 압축
* **출력 데이터:** `new_orleans_pca_32.csv` (`data/embeddings` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil
from pathlib import Path

print("[뉴올리언스] PCA 차원 축소 시작")

# 1. 32차원으로 압축하는 PCA 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 생성된 new_orleans_embeddings 데이터를 바로 압축
embeddings_pca = pca.fit_transform(new_orleans_embeddings)

# 3. 데이터 손실률 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보 보존율: {explained_variance:.2f}%")
print(f"압축된 데이터 크기: {embeddings_pca.shape}")

# 4. 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 안전 저장 (코랩 ➔ 구글 드라이브)
local_csv_path = '/content/new_orleans_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/data/embeddings/new_orleans_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
Path(drive_csv_path).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(local_csv_path, drive_csv_path)

print(f"뉴올리언스 3단계 최종 CSV 저장 완료: {drive_csv_path}")

---
# 4단계. 모델링 및 분석

로지스틱 회귀, CatBoost, PyTorch MLP를 학습하고 성능 평가와 SHAP 분석 산출물을 저장합니다.

원본 노트북: `notebooks/04_modeling_analysis/Modeling.ipynb`


In [ ]:
# 0. 나눔고딕 폰트와 필요한 패키지 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

!pip install catboost shap

## 0. 환경 설정

필요 라이브러리를 불러오고 프로젝트 루트 및 `output/` 저장 경로를 설정합니다.


In [ ]:
# 0. 필요한 라이브러리 import 및 폰트 설정하기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
import shap
from pathlib import Path
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import roc_curve, auc

# 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if Path(font_path).exists():
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    print(f'나눔 폰트를 찾지 못했습니다: {font_path}. 기본 글꼴을 사용합니다.')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지
shap.initjs() # SHAP 자바스크립트 시각화 활성화

# 프로젝트 경로와 산출물 저장 경로
PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
if not PROJECT_ROOT.exists():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다. Team-6 저장소 안에서 노트북을 실행해 주세요.')

FIGURE_DIR = PROJECT_ROOT / 'output' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
REPORT_DIR = PROJECT_ROOT / 'output' / 'reports'
for directory in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def slugify(value):
    return str(value).lower().replace(' ', '_').replace('-', '_')


def save_current_figure(filename):
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print(f'그래프 저장 완료: {path}')


## 1. 평가 함수 정의

세 모델을 같은 기준으로 비교하기 위해 Accuracy, F1, ROC-AUC, classification report, confusion matrix를 한 함수에서 계산합니다. Confusion matrix는 `output/figures/`에 저장합니다.


In [ ]:
# 세 가지 모델을 동일한 기준으로 공정하게 채점하기 위한 통합 성능 측정 함수입니다. 뒤에서 같은 거 3번 쓰는 것보다 함수로 만들면 편할 거 같아 이렇게 해보았습니다.

# 1차적인 성능 지표로서 정확도, 불균형 데이터에 중요한 F1 점수, 모델의 분류 성능인 ROC-AUC 값이 나오며,
# 이후 긍정과 부정별로 정밀도와 재현율을 볼 수 있도록 하고,
# 모델이 어떤 걸 맞추고 틀렸는지 볼 수 있는 혼동 행렬를 그려보았습니다.

def evaluate_model(y_true, y_pred, y_proba, model_name, city_name):
    print(f"\n[{city_name}] {model_name} 모델 성능 평가")
    print(f"정확도    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 점수   : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_proba):.4f}")

    print("\n[정밀도와 재현율]")
    print(classification_report(y_true, y_pred, target_names=['만족(0)', '불만족(1)']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['예측: 만족', '예측: 불만족'],
                yticklabels=['실제: 만족', '실제: 불만족'])
    plt.title(f'{city_name} {model_name} 혼동 행렬')
    plt.ylabel('실제 라벨')
    plt.xlabel('예측 라벨')
    plt.tight_layout()
    save_current_figure(f"model_{slugify(city_name)}_{slugify(model_name)}_confusion_matrix.png")
    plt.show()

## 2. 데이터 로드 및 분할

로컬 `data/processed`와 `data/embeddings` 파일을 우선 사용하고, 로컬 파일이 없을 때만 GitHub raw URL을 fallback으로 사용합니다.


In [ ]:
# 로컬 CSV를 우선 사용하고, 파일이 없으면 GitHub 원격 파일을 예비 경로로 사용합니다.
# target_label은 불만족 리뷰를 1로 두기 위해 is_positive를 반전합니다.

data_sources = {
    "New Orleans": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_new_orleans_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "new_orleans_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_new_orleans_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/new_orleans_pca_32.csv",
    },
    "Philly": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_philly_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "philly_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_philly_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/philly_pca_32.csv",
    },
    "Tucson": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_tucson_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "tucson_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_tucson_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/tucson_pca_32.csv",
    },
}

github_urls = data_sources

def read_csv_with_fallback(local_path, remote_url):
    if Path(local_path).exists():
        print(f'로컬 CSV 불러오기: {local_path}')
        return pd.read_csv(local_path)
    print(f'로컬 CSV가 없어 원격 CSV를 불러옵니다: {remote_url}')
    return pd.read_csv(remote_url)

target_col = 'is_positive'
city_datasets = {}
global_results = {city: {} for city in data_sources.keys()}

for city, sources in data_sources.items():
    df_raw = read_csv_with_fallback(sources["raw_path"], sources["raw_url"])
    df_pca = read_csv_with_fallback(sources["pca_path"], sources["pca_url"])

    df_pca['target_label'] = 1 - df_raw[target_col].values

    X = df_pca.drop(columns=['target_label'])
    y = df_pca['target_label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    city_datasets[city] = (X_train, X_test, y_train, y_test)
    print(f"{city} 데이터 분할 완료 (학습: {X_train.shape[0]}건, 평가: {X_test.shape[0]}건)")


## 3. PyTorch MLP 구조 정의

32차원 PCA 임베딩을 입력으로 받는 MLP와 학습 루프를 정의합니다. 클래스 불균형은 `BCEWithLogitsLoss(pos_weight=...)`로 보정합니다.


In [ ]:
# PyTorch를 활용하여 딥러닝 모델의 아키텍처와 학습 함수를 정의하였습니다.
# 32차원으로 압축된 임베딩 데이터를 입력받아서 긍정/부정 확률을 출력하도록 3개의 층으로 구성하였고,
# 데이터의 클래스 불균형, 즉 불만족 리뷰가 너무 적거나 하는 등의 현상을 해결하기 위해서 불만족 리뷰를 놓쳤을 때 더 큰 가중치를 부여하도록 하는 비용 민감 학습인 BCEWithLogitsLoss를 적용하였습니다.
# 모델 학습이 끝나면 정의한 평가 함수를 호출하고, 추후 시각화를 위해 예측 확률을 저장하도록 하였습니다.

# 데이터셋 클래스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, input_dim=32):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

def train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30):
    train_loader = DataLoader(ReviewDataset(X_train, y_train), batch_size=64, shuffle=True)
    test_loader = DataLoader(ReviewDataset(X_test, y_test), batch_size=64, shuffle=False)
    model = MLP(input_dim=32).to(device)

    count_satisfied = sum(y_train == 0)
    count_dissatisfied = sum(y_train == 1)

    # BCEWithLogitsLoss의 pos_weight는 (부정 Class Count / 긍정 Class Count) 입니다
    pos_weight_val = torch.tensor([count_satisfied / count_dissatisfied], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for batch_X, _ in test_loader:
            probs = torch.sigmoid(model(batch_X.to(device)))
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend((probs >= 0.5).float().cpu().numpy())

    preds_flat = np.array(all_preds).flatten()
    probs_flat = np.array(all_probs).flatten()

    evaluate_model(y_test, preds_flat, probs_flat, "PyTorch MLP", city)
    global_results[city]['dl_proba'] = probs_flat

## 4. 도시별 모델 학습

각 도시에서 Logistic Regression, CatBoost, PyTorch MLP를 학습하고 예측 확률을 `global_results`에 저장합니다.


In [ ]:
# 앞서 분할한 데이터 중 먼저 뉴올리언스 지역의 데이터를 바탕으로 세 가지 모델의 학습 및 평가를 진행하였습니다.
# 학습이 끝난 후, 각 모델의 예측 확률값과 CatBoost 모델 객체는 최종 비교 시각화 및 SHAP 분석을 위해 딕셔너리에 저장하였습니다.

city = "New Orleans"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

In [ ]:
# 필라델피아 지역

city = "Philly"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

In [ ]:
# 투손 지역

city = "Tucson"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

## 5. 결과 시각화 및 해석

도시별 ROC 곡선, SHAP Summary/의존성 그래프, 종합 성능표, 오류 분석 결과를 생성하고 `output/`에 저장합니다.


In [ ]:
# 앞서 저장해둔 예측 결과와 모델 객체를 바탕으로 뉴올리언스 지역의 시각화 리포트를 생성하였습니다.
# 세 모델의 성능을 비교하는 ROC 곡선, 핵심 요인을 분석하는 SHAP 요약 그래프 및 의존성 그래프을 시각화하였습니다.
# 마지막으로 전체 데이터의 SHAP 값을 군집화(Clustering)하는 Heatmap Plot을 추가하여, 개별 리뷰가 아닌 '전체 고객의 불만 유형(거시 패턴)'이 어떻게 세분화되는지 거시적인 마케팅 인사이트를 도출하였습니다.

city = "New Orleans"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('뉴올리언스 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('new_orleans_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("뉴올리언스 피처 중요도")
plt.tight_layout()
save_current_figure('new_orleans_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"뉴올리언스 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('new_orleans_shap_dependence.png')
plt.show()

In [ ]:
# 필라델피아입니다.

city = "Philly"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('필라델피아 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('philly_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("필라델피아 피처 중요도")
plt.tight_layout()
save_current_figure('philly_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"필라델피아 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('philly_shap_dependence.png')
plt.show()


In [ ]:
# 투손입니다.

city = "Tucson"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('투손 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('tucson_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("투손 피처 중요도")
plt.tight_layout()
save_current_figure('tucson_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"투손 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('tucson_shap_dependence.png')
plt.show()

In [ ]:
# 3개 도시와 3가지 모델의 성능 지표(F1 점수, ROC-AUC)를 하나의 그래프로 통합하여 비교 분석하였습니다.
# 전역 딕셔너리에 저장된 예측 확률값을 활용하여 각 환경별 지표를 재산출하고, 이를 바 차트로 시각화하였습니다.

data_rows = []

for city in city_datasets.keys():
    _, _, _, y_test = city_datasets[city]

    lr_proba = global_results[city]['lr_proba']
    cb_proba = global_results[city]['cb_proba']
    dl_probs = global_results[city]['dl_proba']

    lr_pred = (lr_proba >= 0.5).astype(int)
    cb_pred = (cb_proba >= 0.5).astype(int)
    dl_pred = (dl_probs >= 0.5).astype(int)

    for model_name, preds, probas in [("로지스틱 회귀", lr_pred, lr_proba),
                                      ("CatBoost", cb_pred, cb_proba),
                                      ("PyTorch MLP", dl_pred, dl_probs)]:
        f1 = f1_score(y_test, preds)
        auc_val = roc_auc_score(y_test, probas)
        data_rows.append({"City": city, "Model": model_name, "F1 점수": f1, "ROC-AUC": auc_val})

df_metrics = pd.DataFrame(data_rows)
metrics_path = TABLE_DIR / 'model_metrics_summary.csv'
df_metrics.to_csv(metrics_path, index=False)
print(f'성능 요약표 저장 완료: {metrics_path}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(x="City", y="F1 점수", hue="Model", data=df_metrics, ax=axes[0], palette="muted")
axes[0].set_title("도시별 모델 F1 점수 비교")
axes[0].set_ylim(0.7, 1.0)
axes[0].set_xlabel("도시")
axes[0].grid(axis='y', alpha=0.3)

sns.barplot(x="City", y="ROC-AUC", hue="Model", data=df_metrics, ax=axes[1], palette="muted")
axes[1].set_title("도시별 모델 ROC-AUC 비교")
axes[1].set_ylim(0.9, 1.05)
axes[1].set_xlabel("도시")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_current_figure('model_metrics_comparison.png')
plt.show()

In [ ]:
# 모델이 예측에 실패한 원인을 규명하기 위해 도시별 오류 분석(오류 분석)을 수행하였습니다.
# 정답을 맞춘 불만족 리뷰(참 양성)와 만족 리뷰로 오해한 불만족 리뷰(거짓 음성)의 SHAP 값을 대조하여 모델이 분류에 혼선을 겪은 요인을 추적하였습니다.

for city in city_datasets.keys():
    X_train, X_test, y_train, y_test = city_datasets[city]
    cb_model = global_results[city]['cb_model']
    cb_preds = cb_model.predict(X_test)

    explainer = shap.TreeExplainer(cb_model)
    shap_values = explainer.shap_values(X_test)

    tp_idx = np.where((y_test == 1) & (cb_preds == 1))[0]
    fn_idx = np.where((y_test == 1) & (cb_preds == 0))[0]

    if len(fn_idx) > 0:
        tp_shap_mean = np.abs(shap_values[tp_idx]).mean(axis=0)
        fn_shap_mean = np.abs(shap_values[fn_idx]).mean(axis=0)

        df_error = pd.DataFrame({
            'Feature': X_test.columns,
            '정확한 불만 예측(TP)': tp_shap_mean,
            '오류 유발 불만 예측(FN)': fn_shap_mean
        }).sort_values(by='정확한 불만 예측(TP)', ascending=False).head(10)

        error_path = TABLE_DIR / f"error_analysis_{slugify(city)}.csv"
        df_error.to_csv(error_path, index=False)
        print(f'오류 분석표 저장 완료: {error_path}')

        df_error_melt = df_error.melt(id_vars='Feature', var_name='Group', value_name='Mean SHAP')

        plt.figure(figsize=(10, 5))
        sns.barplot(x='Mean SHAP', y='Feature', hue='Group', data=df_error_melt, palette='Set2')
        plt.title(f"[{city}] 예측 성공 및 실패 집단 간 SHAP 기여도 대조")
        plt.xlabel("SHAP 기여도 평균")
        plt.ylabel("핵심 피처")
        plt.tight_layout()
        save_current_figure(f"error_analysis_{slugify(city)}.png")
        plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

# 특정 PCA Component의 고정값(높은 값/낮은 값)에 매핑되는 원본 텍스트를 추출하여 해당 차원의 언어학적/문맥적 의미를 해석하기 위한 역추적 분석 함수입니다.

def analyze_pca_meaning(city_name, pca_feature_name, top_n=5):
    urls = data_sources[city_name]
    df_raw = read_csv_with_fallback(urls["raw_path"], urls["raw_url"])
    df_pca = read_csv_with_fallback(urls["pca_path"], urls["pca_url"])

    text_col = 'text' if 'text' in df_raw.columns else df_raw.select_dtypes(include=[object]).columns[0]

    df_analysis = pd.DataFrame({
        'text': df_raw[text_col],
        'pca_value': df_pca[pca_feature_name]
    })

    top_reviews = df_analysis.sort_values(by='pca_value', ascending=False).head(top_n)
    bottom_reviews = df_analysis.sort_values(by='pca_value', ascending=True).head(top_n)

    print(f"[{city_name}] {pca_feature_name} 차원 해석")

    print(f"\n  [높은 값]")
    for row in top_reviews.itertuples():
        print(f"  ({row.pca_value:.4f}) {row.text[:120]}...")

    print(f"\n  [낮은 값]")
    for row in bottom_reviews.itertuples():
        print(f"  ({row.pca_value:.4f}) {row.text[:120]}...")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    vectorizer = CountVectorizer(stop_words='english', max_features=10)

    try:
        X_top = vectorizer.fit_transform(top_reviews['text'])
        top_words = pd.DataFrame(X_top.toarray(), columns=vectorizer.get_feature_names_out()).sum().sort_values(ascending=False)
        sns.barplot(x=top_words.values, y=top_words.index, ax=axes[0], palette='bone')
        axes[0].set_title(f"{pca_feature_name} (High)")
        axes[0].set_xlabel("빈도")
    except Exception:
        axes[0].text(0.5, 0.5, '데이터 부족', ha='center')

    try:
        X_bottom = vectorizer.fit_transform(bottom_reviews['text'])
        bottom_words = pd.DataFrame(X_bottom.toarray(), columns=vectorizer.get_feature_names_out()).sum().sort_values(ascending=False)
        sns.barplot(x=bottom_words.values, y=bottom_words.index, ax=axes[1], palette='bone')
        axes[1].set_title(f"{pca_feature_name} (Low)")
        axes[1].set_xlabel("빈도")
    except Exception:
        axes[1].text(0.5, 0.5, '데이터 부족', ha='center')

    plt.tight_layout()
    save_current_figure(f"pca_meaning_{slugify(city_name)}_{pca_feature_name}.png")
    plt.show()

In [ ]:
analyze_pca_meaning(city_name="New Orleans", pca_feature_name="roberta_pca_3", top_n=3)

In [ ]:
analyze_pca_meaning(city_name="Philly", pca_feature_name="roberta_pca_3", top_n=3)

In [ ]:
analyze_pca_meaning(city_name="Tucson", pca_feature_name="roberta_pca_3", top_n=3)